In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

os.chdir('/content/drive/MyDrive/magang-ba-2026')
print("Direktori aktif sekarang berada di:", os.getcwd())

Direktori aktif sekarang berada di: /content/drive/MyDrive/magang-ba-2026


**Unduh model sesuai keputusan**

In [3]:
import os
import json

# 1. Pastikan direktori 'models' tersedia
os.makedirs("models", exist_ok=True)

# 2. Menyusun data Model Card sesuai panduan
model_card = {
    "model_id": "crypter70/IndoBERT-Sentiment-Analysis",
    "revision": "main",
    "license": "mit",
    "task": "text-classification",
    "label_mapping": {
        "positive": "positif",
        "neutral": "netral",
        "negative": "negatif"
    },
    "description": "Model IndoBERT fine-tuned untuk analisis sentimen bahasa Indonesia (Opsi B - Siap pakai)"
}

In [4]:
# 3. Menyimpan ke file models/T1_model_card.json
with open("models/T1_model_card.json", "w", encoding="utf-8") as f:
    json.dump(model_card, f, indent=4, ensure_ascii=False)

print("Berhasil membuat dan menyimpan models/T1_model_card.json")

Berhasil membuat dan menyimpan models/T1_model_card.json


**Menjalankan Inferensi T1 pada Validation Set (Opsi B)**

In [6]:
import json
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import pipeline

# 1. Muat dataset & lakukan split secara konsisten
df_comments = pd.read_csv("data/processed/comments_v1.csv")
text_col = 'text_original'
label_col = 'sentiment_label'

df_clean = df_comments.dropna(subset=[text_col, label_col]).copy()
df_clean['reference_label'] = df_clean[label_col]

train_val, test_df = train_test_split(
    df_clean, test_size=0.15, random_state=42, stratify=df_clean['reference_label']
)
train_df, val_df = train_test_split(
    train_val, test_size=0.1764, random_state=42, stratify=train_val['reference_label']
)

In [7]:
# 2. Ambil teks validasi dan label aslinya
texts_val = val_df[text_col].tolist()
true_labels = val_df['reference_label'].astype(str).str.lower().str.strip().tolist()

print(f"Jumlah data validasi (val_df): {len(val_df)}")

Jumlah data validasi (val_df): 145


In [8]:
# 3. Muat Model Card yang baru saja kita buat
with open("models/T1_model_card.json", "r", encoding="utf-8") as f:
    model_card = json.load(f)

In [9]:
# 4. Inisialisasi Pipeline Transformer (Opsi B - Tanpa pelatihan)
clf_t1 = pipeline(
    "text-classification",
    model=model_card["model_id"],
    revision=model_card["revision"],
    device=-1,           # Menggunakan CPU
    truncation=True,
    max_length=128
)

config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  498MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/709k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [10]:
# 5. Jalankan inferensi pada seluruh data validasi
print("\nMenjalankan inferensi T1 (max_length=128) pada CPU...")
outputs_t1 = clf_t1(texts_val, batch_size=16)


Menjalankan inferensi T1 (max_length=128) pada CPU...


In [11]:
# 6. Fungsi pemetaan label dari output Inggris model ke Indonesia sesuai Model Card
def map_english_to_indo(model_label):
    lbl = str(model_label).upper().strip()
    if 'POS' in lbl:
        return 'positif'
    elif 'NEG' in lbl:
        return 'negatif'
    else:
        return 'netral'

preds_t1 = [map_english_to_indo(o["label"]) for o in outputs_t1]
scores_t1 = [o["score"] for o in outputs_t1]

print("Inferensi T1 berhasil diselesaikan!")

Inferensi T1 berhasil diselesaikan!


**Mengukur Durasi, Menghitung Metrik, dan Menyimpan Hasil T1**

In [12]:
import time
import json
import os
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

# 1. Mengukur durasi waktu inferensi pada validation set
start_time = time.time()
_ = clf_t1(texts_val, batch_size=16)
end_time = time.time()
inference_duration = end_time - start_time
print(f"Durasi inferensi {len(texts_val)} data di CPU: {inference_duration:.4f} detik")

Durasi inferensi 145 data di CPU: 43.6101 detik


In [13]:
# 2. Menghitung metrik evaluasi
acc_t1 = accuracy_score(true_labels, preds_t1)
report_t1 = classification_report(true_labels, preds_t1, output_dict=True, zero_division=0)

print(f"Akurasi T1: {acc_t1:.4f}")

Akurasi T1: 0.4207


In [14]:
# 3. Menyimpan hasil detail ke results/T1_val.json
os.makedirs("results", exist_ok=True)
t1_results = {
    "experiment_id": "T1_indobert",
    "model_id": model_card["model_id"],
    "max_length": 128,
    "val_samples": len(texts_val),
    "accuracy": acc_t1,
    "inference_duration_seconds": inference_duration,
    "classification_report": report_t1
}

with open("results/T1_val.json", "w", encoding="utf-8") as f:
    json.dump(t1_results, f, indent=4, ensure_ascii=False)
print("Berhasil menyimpan: results/T1_val.json")

Berhasil menyimpan: results/T1_val.json


In [15]:
# 4. Menambahkan/Memperbarui baris ringkasan di results/experiments.csv
experiments_path = "results/experiments.csv"

if os.path.exists(experiments_path):
    df_exp = pd.read_csv(experiments_path)
    # Hapus baris lama T1 jika sudah ada agar tidak duplikat
    df_exp = df_exp[df_exp['experiment_id'] != 'T1_indobert']
else:
    df_exp = pd.DataFrame(columns=["experiment_id", "model_type", "max_features", "ngram_range", "train_samples", "val_samples", "val_accuracy"])

new_row = pd.DataFrame([{
    "experiment_id": "T1_indobert",
    "model_type": "IndoBERT (max_length=128)",
    "max_features": "N/A (Tokenizer)",
    "ngram_range": "N/A",
    "train_samples": len(train_df),
    "val_samples": len(val_df),
    "val_accuracy": acc_t1
}])

df_updated = pd.concat([df_exp, new_row], ignore_index=True)
df_updated.to_csv(experiments_path, index=False)
print("Berhasil memperbarui: results/experiments.csv")

Berhasil memperbarui: results/experiments.csv
